In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

REPORT_IMG_DIR = Path(r"C:\Users\ferra\MIC\1r_any_UNICAS\2n_Semestre\Image_Processing_and_Analysis\project\MIC_project\Proposal_StenosisDetection\Report_selected_images")

img_paths = sorted(REPORT_IMG_DIR.glob("*.png")) + sorted(REPORT_IMG_DIR.glob("*.bmp"))
print(img_paths)

imgs = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in img_paths]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
labels = ['a)', 'b)', 'c)']
for ax, img, label in zip(axes, imgs, labels):
    ax.imshow(img, cmap='gray')
    ax.text(0.02, 0.97, label, transform=ax.transAxes,
            fontsize=14, fontweight='bold', color='white',
            va='top', ha='left')
    ax.axis('off')

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / 'step_original.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import sys
from src.config import *
sys.path.append(ROOT_DIR)
from src.utils import parse_stenosis_xml, draw_bboxes

xml_paths = [REPORT_IMG_DIR / (p.stem + '.xml') for p in img_paths]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, img, xml_path in zip(axes, imgs, xml_paths):
    boxes     = parse_stenosis_xml(str(xml_path))
    img_boxes = draw_bboxes(img, boxes)
    img_rgb   = cv2.cvtColor(img_boxes, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.axis('off')

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / 'step_gt_bboxes.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np

SERIES_DIRS = {
    "14_021_1": REPORT_IMG_DIR / "14_021_1",
    "14_088_2": REPORT_IMG_DIR / "14_088_2",
    "14_092_2": REPORT_IMG_DIR / "14_092_2",
}

def compute_serie_mean(serie_dir: Path) -> np.ndarray:
    stack = [
        cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).astype(np.float32)
        for p in sorted(serie_dir.glob("*.png"))
        if cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) is not None
    ]
    return np.mean(stack, axis=0)

def temporal_subtract(image: np.ndarray, mean_image: np.ndarray) -> np.ndarray:
    residual = image.astype(np.float32) - mean_image
    r_min, r_max = residual.min(), residual.max()
    if r_max > r_min:
        return ((residual - r_min) / (r_max - r_min) * 255).astype(np.uint8)
    return np.zeros_like(image)

means, subtracted_frames = [], []
for serie_name, serie_dir in SERIES_DIRS.items():
    mean_img = compute_serie_mean(serie_dir)
    rep_path = next(REPORT_IMG_DIR.glob(f"{serie_name}_*.png"))
    rep_frame = cv2.imread(str(rep_path), cv2.IMREAD_GRAYSCALE)
    means.append(mean_img.astype(np.uint8))
    subtracted_frames.append(temporal_subtract(rep_frame, mean_img))

# 3-row figure: mean | original frame | subtracted frame
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

row_names = ["Mean image", "Original frame", "Subtracted frame"]

for row_idx, (row_imgs, row_name) in enumerate(zip([means, imgs, subtracted_frames], row_names)):
    for col_idx, (ax, img) in enumerate(zip(axes[row_idx], row_imgs)):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        if col_idx == 0:
            ax.text(-0.05, 0.5, row_name,
                    transform=ax.transAxes,
                    fontsize=13, fontweight="bold",
                    va="center", ha="right",
                    rotation=90)

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_temporal_subtraction.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from src.preprocessing import subtract_background, apply_nlm, apply_clahe

KERNEL_SIZE = 81
H           = 8
CLIP_LIMIT  = 4.0

# Build all pipeline stages for each representative frame
stage_results = [[], [], [], []]  # temporal_sub | bg_sub | nlm | clahe
for s1, img in zip(subtracted_frames, imgs):
    s2 = subtract_background(s1, kernel_size=KERNEL_SIZE)
    s3 = apply_nlm(s2, h=H, template_window=7, search_window=21)
    s4 = apply_clahe(s3, clip_limit=CLIP_LIMIT)

    for stage_list, result in zip(stage_results, [s1, s2, s3, s4]):
        stage_list.append(result)

row_names = [
    "Subtracted frame",
    f"Gaussian BG subtraction (Kernel size: {KERNEL_SIZE})",
    f"NLM denoising (Filter stength: {H})",
    f"CLAHE (Clip limit: {CLIP_LIMIT})",
]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for row_idx, (row_imgs, row_name) in enumerate(zip(stage_results, row_names)):
    for col_idx, (ax, img) in enumerate(zip(axes[row_idx], row_imgs)):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        if col_idx == 0:
            ax.text(-0.05, 0.5, row_name,
                    transform=ax.transAxes,
                    fontsize=13, fontweight="bold",
                    va="center", ha="right",
                    rotation=90)

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_enhancement_pipeline.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col_idx, (ax, img, rep_path) in enumerate(zip(axes, stage_results[3], img_paths)):
    xml_path = rep_path.with_suffix(".xml")
    boxes    = parse_stenosis_xml(str(xml_path))
    img_anno = draw_bboxes(img, boxes)
    img_rgb  = cv2.cvtColor(img_anno, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.axis("off")

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_CLAHE_pipeline.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from src.vessel_segmentation import apply_frangi, apply_sato, apply_meijering

SCALE_MIN = 4
SCALE_MAX = 15

filter_results = [[], [], []]  # frangi | sato | meijering
filter_fns     = [apply_frangi, apply_sato, apply_meijering]

for img in stage_results[3]:
    for filter_list, filter_fn in zip(filter_results, filter_fns):
        filter_list.append(filter_fn(img, scale_min=SCALE_MIN, scale_max=SCALE_MAX, black_ridges=True))

row_names = ["Frangi", "Sato", "Meijering"]

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for row_idx, (row_imgs, row_name) in enumerate(zip(filter_results, row_names)):
    for col_idx, (ax, img) in enumerate(zip(axes[row_idx], row_imgs)):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        if col_idx == 0:
            ax.text(-0.05, 0.5, row_name,
                    transform=ax.transAxes,
                    fontsize=13, fontweight="bold",
                    va="center", ha="right",
                    rotation=90)

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_vesselness_filters.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from src.vessel_segmentation import apply_frangi, apply_cleaned_hysteresis
from skimage.morphology import skeletonize
from skimage.measure import label

SCALE_MIN   = 4
SCALE_MAX   = 15
LOW_THRESH  = 0.02
HIGH_THRESH = 0.18
MIN_SIZE    = 200
DIST_MAX    = 25.0

def optimize_skeleton(bool_mask):
    skeleton_raw = skeletonize(bool_mask)
    labeled_skel, num_features = label(skeleton_raw, return_num=True)

    if num_features > 0:
        pixel_counts = np.bincount(labeled_skel.ravel())
        pixel_counts[0] = 0
        sorted_indices = np.argsort(pixel_counts)
        idx_first  = sorted_indices[-1]
        px_first   = pixel_counts[idx_first]

        if num_features >= 2:
            idx_second = sorted_indices[-2]
            px_second  = pixel_counts[idx_second]
            valid = [idx_first, idx_second] if px_second > 0.2 * px_first else [idx_first]
        else:
            valid = [idx_first]

        skeleton_orig = np.isin(labeled_skel, valid)
    else:
        skeleton_orig = skeleton_raw.copy()

    kernel = np.array([[1,1,1],[1,0,1],[1,1,1]], dtype=np.uint8)
    neighbor_count = cv2.filter2D(skeleton_orig.astype(np.uint8), -1, kernel)
    endpoints_mask = (neighbor_count == 1) & skeleton_orig

    y_ends, x_ends  = np.where(endpoints_mask)
    endpoints       = list(zip(x_ends, y_ends))
    bridge_layer    = np.zeros_like(skeleton_orig, dtype=np.uint8)

    for i in range(len(endpoints)):
        for j in range(i + 1, len(endpoints)):
            pt1, pt2 = endpoints[i], endpoints[j]
            dist = np.sqrt((pt1[0]-pt2[0])**2 + (pt1[1]-pt2[1])**2)
            if 2.0 < dist <= DIST_MAX:
                cv2.line(bridge_layer, pt1, pt2, color=1, thickness=1)

    return skeleton_orig | (bridge_layer > 0)

# Build all four stages for each representative frame
clahe_imgs, frangi_maps, masks, skeletons = [], [], [], []

for img in stage_results[3]:
    frangi_map = apply_frangi(img, scale_min=SCALE_MIN, scale_max=SCALE_MAX, black_ridges=True)
    bool_mask  = apply_cleaned_hysteresis(frangi_map, low_thresh=LOW_THRESH, high_thresh=HIGH_THRESH, min_size=MIN_SIZE)
    skeleton   = optimize_skeleton(bool_mask)

    clahe_imgs.append(img)
    frangi_maps.append(frangi_map)
    masks.append(bool_mask)
    skeletons.append(skeleton)

row_names = ["CLAHE", "Frangi vesselness", "Hysteresis mask", "Skeleton"]
all_rows  = [clahe_imgs, frangi_maps, masks, skeletons]

fig, axes = plt.subplots(4, 3, figsize=(15, 20))

for row_idx, (row_imgs, row_name) in enumerate(zip(all_rows, row_names)):
    for col_idx, (ax, img) in enumerate(zip(axes[row_idx], row_imgs)):
        ax.imshow(img, cmap="gray")
        ax.axis("off")
        if col_idx == 0:
            ax.text(-0.05, 0.5, row_name,
                    transform=ax.transAxes,
                    fontsize=13, fontweight="bold",
                    va="center", ha="right",
                    rotation=90)

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_segmentation_pipeline.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for col_idx, (ax, clahe_img, mask, skel, rep_path) in enumerate(
        zip(axes, clahe_imgs, masks, skeletons, img_paths)):

    # Base: CLAHE image in BGR
    base = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR).astype(np.float32)

    # Green mask overlay (BGR: [0, 255, 0])
    green_layer = np.zeros_like(base)
    green_layer[mask] = [0, 255, 0]
    base = cv2.addWeighted(base, 1.0, green_layer, 0.3, 0)

    # Blue skeleton — hard-write at full opacity after converting to uint8
    base_u8 = np.clip(base, 0, 255).astype(np.uint8)
    base_u8[skel > 0] = [255, 0, 0]

    # Red ground truth bboxes (BGR: [0, 0, 255])
    boxes = parse_stenosis_xml(str(rep_path.with_suffix(".xml")))
    for box in boxes:
        cv2.rectangle(base_u8,
                  (box['xmin'], box['ymin']),
                  (box['xmax'], box['ymax']),
                  color=(0, 0, 255), thickness=2)

    # Convert BGR → RGB only at display time
    ax.imshow(cv2.cvtColor(base_u8, cv2.COLOR_BGR2RGB))
    ax.axis("off")

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_overlay.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from src.feature_extraction import (
    extract_oriented_rois, label_oriented_rois, crop_oriented_roi,
    _order_skeleton_points_fast
)

TOTAL_ROIS  = 100
RECT_WIDTH  = 100
RECT_HEIGHT = 50

# Color coding (BGR): green=negative, red=positive, yellow=undetermined, magenta=GT
COLORS = {1: (0, 0, 255), 0: (0, 255, 0), -1: (0, 255, 255)}
GT_COLOR = (255, 0, 255)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, clahe_img, skel, rep_path in zip(axes, clahe_imgs, skeletons, img_paths):

    gt_boxes      = parse_stenosis_xml(str(rep_path.with_suffix(".xml")))
    img_shape     = clahe_img.shape[:2]
    oriented_rois = extract_oriented_rois(skel, img_shape,
                                          total_rois=TOTAL_ROIS,
                                          rect_width=RECT_WIDTH,
                                          rect_height=RECT_HEIGHT)

    # GT-centred ROIs (label = 1), matching the pipeline in the notebook
    ordered      = _order_skeleton_points_fast(skel)
    y_sk, x_sk   = np.where(skel > 0)
    skel_pts     = np.column_stack([x_sk, y_sk]) if len(x_sk) > 0 else np.empty((0, 2))

    for gt_box in gt_boxes:
        cx_gt = (gt_box['xmin'] + gt_box['xmax']) // 2
        cy_gt = (gt_box['ymin'] + gt_box['ymax']) // 2

        if len(skel_pts) > 0:
            dists      = np.sum((skel_pts - np.array([cx_gt, cy_gt])) ** 2, axis=1)
            nearest_pt = tuple(skel_pts[np.argmin(dists)])
            try:
                nearest_idx = ordered.index(nearest_pt)
                tw = 10
                i0 = max(0, nearest_idx - tw)
                i1 = min(len(ordered) - 1, nearest_idx + tw)
                dx = ordered[i1][0] - ordered[i0][0]
                dy = ordered[i1][1] - ordered[i0][1]
                if dx < 0: dx, dy = -dx, -dy
                angle_deg = float(np.degrees(np.arctan2(dy, dx))) % 180.0 if (dx or dy) else 0.0
                cx, cy = nearest_pt
            except ValueError:
                angle_deg, cx, cy = 0.0, cx_gt, cy_gt
        else:
            angle_deg, cx, cy = 0.0, cx_gt, cy_gt

        oriented_rois.append({'center': (cx, cy), 'angle': angle_deg,
                               'width': RECT_WIDTH, 'height': RECT_HEIGHT,
                               'gt_centered': True})

    labels = label_oriented_rois(oriented_rois, gt_boxes)

    # Draw everything on a BGR canvas
    canvas = cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR)

    for roi, lbl in zip(oriented_rois, labels):
        thickness = 2 if roi.get('gt_centered') else 1
        cx, cy    = roi['center']
        angle_rad = np.radians(roi['angle'])
        cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
        hw, hh    = RECT_WIDTH / 2, RECT_HEIGHT / 2

        # Four corners of the oriented rectangle
        corners = np.array([
            [cx + cos_a * hw - sin_a * hh, cy + sin_a * hw + cos_a * hh],
            [cx - cos_a * hw - sin_a * hh, cy - sin_a * hw + cos_a * hh],
            [cx - cos_a * hw + sin_a * hh, cy - sin_a * hw - cos_a * hh],
            [cx + cos_a * hw + sin_a * hh, cy + sin_a * hw - cos_a * hh],
        ], dtype=np.int32)

        cv2.polylines(canvas, [corners], isClosed=True,
                      color=COLORS.get(lbl, (128, 128, 128)), thickness=thickness)

    # GT bboxes on top in magenta
    for box in gt_boxes:
        cv2.rectangle(canvas,
                      (box['xmin'], box['ymin']),
                      (box['xmax'], box['ymax']),
                      color=GT_COLOR, thickness=2)

    ax.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    ax.axis("off")

plt.tight_layout()
plt.savefig(REPORT_IMG_DIR / "results" / "step_roi_proposal.png",
            dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
import random as python_random

fig, axes = plt.subplots(3, 4, figsize=(12, 5))

for row_idx, (clahe_img, skel, rep_path) in enumerate(zip(clahe_imgs, skeletons, img_paths)):

    gt_boxes      = parse_stenosis_xml(str(rep_path.with_suffix(".xml")))
    img_shape     = clahe_img.shape[:2]
    oriented_rois = extract_oriented_rois(skel, img_shape,
                                          total_rois=TOTAL_ROIS,
                                          rect_width=RECT_WIDTH,
                                          rect_height=RECT_HEIGHT)

    # GT-centred ROIs (tagged, excluded from stenotic sampling)
    ordered    = _order_skeleton_points_fast(skel)
    y_sk, x_sk = np.where(skel > 0)
    skel_pts   = np.column_stack([x_sk, y_sk]) if len(x_sk) > 0 else np.empty((0, 2))

    for gt_box in gt_boxes:
        cx_gt = (gt_box['xmin'] + gt_box['xmax']) // 2
        cy_gt = (gt_box['ymin'] + gt_box['ymax']) // 2
        if len(skel_pts) > 0:
            dists      = np.sum((skel_pts - np.array([cx_gt, cy_gt])) ** 2, axis=1)
            nearest_pt = tuple(skel_pts[np.argmin(dists)])
            try:
                nearest_idx = ordered.index(nearest_pt)
                tw = 10
                i0 = max(0, nearest_idx - tw)
                i1 = min(len(ordered) - 1, nearest_idx + tw)
                dx = ordered[i1][0] - ordered[i0][0]
                dy = ordered[i1][1] - ordered[i0][1]
                if dx < 0: dx, dy = -dx, -dy
                angle_deg = float(np.degrees(np.arctan2(dy, dx))) % 180.0 if (dx or dy) else 0.0
                cx, cy = nearest_pt
            except ValueError:
                angle_deg, cx, cy = 0.0, cx_gt, cy_gt
        else:
            angle_deg, cx, cy = 0.0, cx_gt, cy_gt

        oriented_rois.append({'center': (cx, cy), 'angle': angle_deg,
                               'width': RECT_WIDTH, 'height': RECT_HEIGHT,
                               'gt_centered': True})

    labels = label_oriented_rois(oriented_rois, gt_boxes)

    # Separate pools: healthy from skeleton-sampled only, stenotic from skeleton-sampled only
    healthy_patches  = []
    stenotic_patches = []

    for roi, lbl in zip(oriented_rois, labels):
        if roi.get('gt_centered'):
            continue
        patch = crop_oriented_roi(clahe_img, roi)
        if patch.shape != (RECT_HEIGHT, RECT_WIDTH):
            continue
        if lbl == 0:
            healthy_patches.append(patch)
        elif lbl == 1:
            stenotic_patches.append(patch)

    # Randomly sample 3 healthy and 1 stenotic
    selected_healthy  = python_random.sample(healthy_patches,  min(3, len(healthy_patches)))
    selected_stenotic = python_random.sample(stenotic_patches, min(1, len(stenotic_patches)))

    # Fill row
    col_titles = ["Healthy", "Healthy", "Healthy", "Stenotic"]
    col_colors = ["green", "green", "green", "red"]
    patches    = selected_healthy + selected_stenotic

    for col_idx in range(4):
        axes[row_idx, 0].text(-0.05, 0.5, f"Patient {['021', '088', '092'][row_idx]}",
                          transform=axes[row_idx, 0].transAxes,
                          fontsize=10, fontweight="bold",
                          va="center", ha="right", rotation=90)
        ax = axes[row_idx, col_idx]
        if col_idx < len(patches):
            ax.imshow(patches[col_idx], cmap="gray")
        else:
            ax.set_facecolor("black")
        ax.axis("off")
        if row_idx == 0:
            ax.set_title(col_titles[col_idx], fontsize=11,
                         fontweight="bold", color=col_colors[col_idx])

plt.subplots_adjust(wspace=0.03, hspace=0.03)
plt.savefig(REPORT_IMG_DIR / "results" / "step_roi_samples.png",
            dpi=300, bbox_inches="tight")
plt.show()